# Download

With this cookbook, one can download one or more CM, at a specified version in a given folder

In [ ]:
# Initialize your connection to Jinko

from jinko import JinkoClient

# This function ensures that authentication is correct
# It it also possible to override the base url by passing baseUrl=...
client = JinkoClient()
client.auth_check()

In [ ]:
# Cookbook specifics imports
import json
import os

# Cookbook specifics constants:
# put here the constants that are specific to your cookbook like
# the reference to the Jinko items, the name of the model, etc.

# folder_id can be retrieved in the url, pattern is `https://jinko.ai/project/<project_id>?labels=<folder_id>`
folder_id = "50c56cbf-1215-4387-9723-e1538ae1eee0"

resources_dir = os.path.normpath("./resources/outputs/download_models")
if not os.path.exists(resources_dir):
    os.makedirs(resources_dir)


def build_model_payload(model):
    return {
        "model": model.content().model_dump(
            mode="json",
            by_alias=True,
            exclude_none=True,
        ),
        "solvingOptions": model.get_solving_options().model_dump(
            mode="json",
            by_alias=True,
            exclude_none=True,
        ),
    }


def save_model_payload(model, directory):
    output_path = os.path.join(
        directory,
        f"{model.name.replace('/', '_')}.json",
    )
    with open(output_path, "w") as f:
        json.dump(build_model_payload(model), f, indent=2)
    print(f"Saved to: {output_path}")
    return output_path

## Step 1 : Display all available models

In [ ]:
# List all models in a particular folder
models = list(client.iter_models(folder=folder_id))
models_by_name = {model.name: model for model in models}
model_dict = {
    model.name: {
        "sid": model.sid,
        "revision": model.version.revision if model.version else None,
        "url": model.url,
    }
    for model in models
}

print("All models in the folder:")
print(json.dumps(model_dict, indent=4))

## Step 2 : Pick one of the methods below to download the models

### download all models of the folder

In [ ]:
for model_name in model_dict:
    print("Downloading model " + str(model_name) + ":")
    save_model_payload(models_by_name[model_name], resources_dir)

### download a specific model 

In [ ]:
MODEL_NAME = "simple tumor model"

print("Downloading model " + str(MODEL_NAME) + ":")
simple_tumor_model = models_by_name[MODEL_NAME]
save_model_payload(simple_tumor_model, resources_dir)

### download a specific model at a given version

list all versions of a model

In [ ]:
MODEL_NAME = "simple tumor model"
simple_tumor_model = client.get_model(model_dict[MODEL_NAME]["sid"])

for version in simple_tumor_model.versions.iter():
    print(
        {
            "revision": version.revision,
            "label": version.label,
            "is_latest": version.is_latest,
            "created_at": version.created_at,
        }
    )

download the first version

In [ ]:
MODEL_REVISION = 1  # picks the first model version

print("Downloading model " + str(MODEL_NAME) + ":")
simple_tumor_model = client.get_model(
    model_dict[MODEL_NAME]["sid"],
    revision=MODEL_REVISION,
)
save_model_payload(simple_tumor_model, resources_dir)